# P4a -- Statistical rigor, land cover @ 100% budget (3 seeds)

**Run this on Kaggle** (Settings -> Accelerator -> GPU T4 x2 or P100; Settings -> Internet -> On).
Continuation of P0/P0b/P3a (all done 2026-10-02/03). See
`Geospatial AI for Public Health -- AI Paper Track (Phases P0-P6) -- Plan.md` for the
full track.

**Scope, stated plainly before any results exist:** the plan's full P4 ask is 3-5
seeds across *every* model x task x budget combination from P0 through P3b -- land
cover (Prithvi/U-Net/Clay x 25/50/100%) and flood/SAR (U-Net/Clay x 25/50/100%), 15
unique configurations. At the real per-run times already observed in this track
(15 min to 66 min each), that full sweep is on the order of 15-25+ hours of GPU time.
This notebook deliberately does **one bounded slice**: 3 seeds x 3 models at the
land-cover task's full (100%) budget -- the paper's central P0/P0b headline
comparison, and the single highest-stakes claim to put real error bars on. The
remaining 12 configurations (both tasks' 25%/50% budgets, and the flood/SAR task's
100% budget) are explicitly **not** attempted here; running them is the same pattern
repeated, not a new engineering problem, and is left as documented follow-up rather
than silently reduced in scope without saying so.

**A real, honest gap this phase also closes:** none of P0/P0b/P3a's original runs
called `pl.seed_everything()` -- their single numbers already on record are *not* one
of three seeds in a controlled sense, just whatever PyTorch's default unseeded
initialization happened to produce. This notebook runs 3 explicitly-seeded repeats
(seeds 0, 1, 2) fresh, rather than treating an unseeded historical run as "seed 0."

**Robustness:** each (model, seed) run is wrapped so a late failure doesn't lose
earlier results -- the comparison CSV is written incrementally after every single run.


## 1. Setup

In [ ]:
!pip install -q "numpy>=2.2" terratorch==1.2.13 lightning segmentation-models-pytorch pystac-client planetary-computer stackstac rioxarray huggingface_hub timm einops torchmetrics
!python -c "import numpy, torch; print('numpy', numpy.__version__, '| torch', torch.__version__)"

!rm -rf repo && git clone -q https://github.com/khalilurrrahmanridoykhan/geohealth-risk-mapping.git repo
!rm -rf clay_repo && git clone -q https://github.com/Clay-foundation/model.git clay_repo
import sys
sys.path.insert(0, 'repo')
sys.path.insert(0, 'clay_repo')


**Running the real pipeline as a subprocess, not inline** -- same verified reason as
every earlier phase in this track (terratorch's numpy>=2.2 requirement vs. Kaggle's
stale-at-startup numpy).

## 2. Write the real pipeline to a script (runs as a subprocess, see note above)

In [ ]:
%%writefile run_p4a.py
import sys, traceback
sys.path.insert(0, 'repo')
sys.path.insert(0, 'clay_repo')

import numpy as np
import torch
import torch.nn.functional as F
import lightning.pytorch as pl
import segmentation_models_pytorch as smp
import terratorch  # noqa: F401
from torch.utils.data import DataLoader, Dataset
import pandas as pd

from src.imagery import get_imagery
from src.landcover import reclassify_worldcover, WATER, BUILT_UP
from src.tiling import tile_array, assign_tile_split

from claymodel.finetune.segment.factory import Segmentor

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', DEVICE)
assert DEVICE == 'cuda', 'this notebook needs a GPU runtime -- check Settings > Accelerator'

CLASS_NAMES = ['other', 'water', 'built_up']
TILE_SIZE = 256
PRITHVI_BANDS = ['B02', 'B03', 'B04', 'B08', 'B11', 'B12']
AOI = (90.30, 23.75, 90.55, 23.95)  # same Dhaka-area tile as P0/P0b/P3a

composite, meta = get_imagery(AOI, '2026-01-01/2026-02-28', sensor='sentinel-2',
                               cloud_cover_max=60, max_scenes=1, resolution=10,
                               bands=PRITHVI_BANDS)
import pystac_client, planetary_computer, stackstac
catalog = pystac_client.Client.open('https://planetarycomputer.microsoft.com/api/stac/v1', modifier=planetary_computer.sign_inplace)
wc_search = catalog.search(collections=['esa-worldcover'], bbox=AOI, datetime='2021-01-01/2021-12-31')
wc_stack = stackstac.stack(list(wc_search.items()), assets=['map'], bounds_latlon=AOI, resolution=10, epsg=meta['epsg'])
worldcover = wc_stack.compute().isel(time=0, band=0).values
labels_4class = reclassify_worldcover(worldcover)
label_map = np.zeros_like(labels_4class, dtype='int64')
label_map[labels_4class == WATER] = 1
label_map[labels_4class == BUILT_UP] = 2

feat_tiles, coords = tile_array(np.nan_to_num(composite.values), tile_size=TILE_SIZE)
label_tiles, _ = tile_array(label_map, tile_size=TILE_SIZE)
test_flags = assign_tile_split(coords, width=composite.shape[-1], test_fraction=0.3)
train_feats = [f for f, t in zip(feat_tiles, test_flags) if not t]
train_labels = [l for l, t in zip(label_tiles, test_flags) if not t]
test_feats = [f for f, t in zip(feat_tiles, test_flags) if t]
test_labels = [l for l, t in zip(label_tiles, test_flags) if t]
print(f'{len(train_feats)} train tiles, {len(test_feats)} held-out test tiles (same split as P0/P0b/P3a)')

results = []

def save_results():
    pd.DataFrame(results).to_csv('comparison_p4a.csv', index=False)
    print('--- comparison_p4a.csv so far ---')
    print(pd.DataFrame(results).to_string(index=False))

def tidy_terratorch(metrics, model_name, seed):
    row = {'model': model_name, 'seed': seed}
    for k, v in metrics[0].items():
        if 'iou' in k.lower():
            name = k.split('/')[-1].lower()
            name = name[4:] if name.startswith('iou_') else name
            row[name] = round(v, 3)
    return row

CONSTANT_SCALE = 1e-4
CLAY_MEAN = np.array([1105., 1355., 1552., 2743., 2388., 1835.], dtype='float32')
CLAY_STD = np.array([1809., 1757., 1888., 1742., 1470., 1379.], dtype='float32')
CLAY_WAVES = torch.tensor([0.493, 0.56, 0.665, 0.842, 1.61, 2.19], dtype=torch.float32)
CLAY_GSD = torch.tensor(10.0)

band_mean = (np.stack(train_feats) * CONSTANT_SCALE).mean(axis=(0, 2, 3))
band_std = (np.stack(train_feats) * CONSTANT_SCALE).std(axis=(0, 2, 3)) + 1e-6


def run_prithvi(seed):
    pl.seed_everything(seed, workers=True)

    class PDS(Dataset):
        def __init__(self, feats, labels):
            self.feats, self.labels = feats, labels
        def __len__(self):
            return len(self.feats)
        def __getitem__(self, idx):
            x = (self.feats[idx] * CONSTANT_SCALE - band_mean[:, None, None]) / band_std[:, None, None]
            return {'image': torch.from_numpy(x.astype('float32')), 'mask': torch.from_numpy(self.labels[idx])}

    train_loader = DataLoader(PDS(train_feats, train_labels), batch_size=8, shuffle=True, num_workers=2)
    test_loader = DataLoader(PDS(test_feats, test_labels), batch_size=8, num_workers=2)

    model = terratorch.tasks.SemanticSegmentationTask(
        model_factory='EncoderDecoderFactory',
        model_args={
            'backbone': 'prithvi_eo_v2_300_tl', 'backbone_pretrained': True,
            'backbone_img_size': TILE_SIZE,
            'backbone_bands': ['BLUE', 'GREEN', 'RED', 'NIR_BROAD', 'SWIR_1', 'SWIR_2'],
            'backbone_coords_encoding': [],
            'necks': [{'name': 'SelectIndices', 'indices': [5, 11, 17, 23]},
                      {'name': 'ReshapeTokensToImage'}, {'name': 'LearnedInterpolateToPyramidal'}],
            'decoder': 'UNetDecoder', 'decoder_channels': [512, 256, 128, 64],
            'head_dropout': 0.1, 'num_classes': 3,
        },
        loss='dice', optimizer='AdamW', lr=1e-4, optimizer_hparams={'weight_decay': 0.1},
        scheduler='ReduceLROnPlateau', scheduler_hparams={'factor': 0.5, 'patience': 5},
        freeze_backbone=False, freeze_decoder=False, class_names=CLASS_NAMES, plot_on_val=False,
    )
    trainer = pl.Trainer(accelerator='gpu', devices=1, precision='16-mixed', max_epochs=100,
                          log_every_n_steps=5, callbacks=[pl.callbacks.EarlyStopping(monitor='val/loss', patience=15)],
                          default_root_dir=f'output/p4a_prithvi_seed{seed}/')
    trainer.fit(model, train_dataloaders=train_loader, val_dataloaders=test_loader)
    metrics = trainer.test(model, dataloaders=test_loader)
    return tidy_terratorch(metrics, 'prithvi_eo_v2_300_tl', seed)


def run_unet(seed):
    pl.seed_everything(seed, workers=True)

    class UDS(Dataset):
        def __init__(self, feats, labels):
            self.feats, self.labels = feats, labels
        def __len__(self):
            return len(self.feats)
        def __getitem__(self, idx):
            x = (self.feats[idx] * CONSTANT_SCALE - band_mean[:, None, None]) / band_std[:, None, None]
            return {'image': torch.from_numpy(x.astype('float32')), 'mask': torch.from_numpy(self.labels[idx])}

    train_loader = DataLoader(UDS(train_feats, train_labels), batch_size=8, shuffle=True, num_workers=2)
    test_loader = DataLoader(UDS(test_feats, test_labels), batch_size=8, num_workers=2)

    class UNetBaseline(pl.LightningModule):
        def __init__(self, in_channels, num_classes):
            super().__init__()
            self.model = smp.Unet(encoder_name='resnet34', encoder_weights=None, in_channels=in_channels, classes=num_classes)
            self.dice = smp.losses.DiceLoss(mode='multiclass')
        def forward(self, x):
            return self.model(x)
        def _step(self, batch, stage):
            logits = self(batch['image'])
            loss = self.dice(logits, batch['mask'])
            self.log(f'{stage}/loss', loss, prog_bar=(stage != 'train'))
            return logits, loss
        def training_step(self, batch, _):
            _, loss = self._step(batch, 'train')
            return loss
        def validation_step(self, batch, _):
            self._step(batch, 'val')
        def test_step(self, batch, _):
            logits, _ = self._step(batch, 'test')
            preds = logits.argmax(dim=1)
            for c, name in enumerate(CLASS_NAMES):
                p, t = preds == c, batch['mask'] == c
                union = (p | t).sum().float()
                iou = (p & t).sum().float() / union if union > 0 else torch.tensor(1.0, device=p.device)
                self.log(f'test/iou_{name}', iou)
        def configure_optimizers(self):
            optimizer = torch.optim.AdamW(self.parameters(), lr=1e-4, weight_decay=0.1)
            scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, factor=0.5, patience=5)
            return {'optimizer': optimizer, 'lr_scheduler': {'scheduler': scheduler, 'monitor': 'val/loss'}}

    model = UNetBaseline(in_channels=len(PRITHVI_BANDS), num_classes=3)
    trainer = pl.Trainer(accelerator='gpu', devices=1, precision='16-mixed', max_epochs=100,
                          log_every_n_steps=5, callbacks=[pl.callbacks.EarlyStopping(monitor='val/loss', patience=15)],
                          default_root_dir=f'output/p4a_unet_seed{seed}/')
    trainer.fit(model, train_dataloaders=train_loader, val_dataloaders=test_loader)
    metrics = trainer.test(model, dataloaders=test_loader)
    return tidy_terratorch(metrics, 'unet_resnet34_scratch', seed)


def run_clay(seed):
    pl.seed_everything(seed, workers=True)

    class CDS(Dataset):
        def __init__(self, feats, labels):
            self.feats, self.labels = feats, labels
        def __len__(self):
            return len(self.feats)
        def __getitem__(self, idx):
            x = (self.feats[idx] - CLAY_MEAN[:, None, None]) / CLAY_STD[:, None, None]
            return {'pixels': torch.from_numpy(x.astype('float32')), 'label': torch.from_numpy(self.labels[idx]),
                    'time': torch.zeros(4), 'latlon': torch.zeros(4)}

    train_loader = DataLoader(CDS(train_feats, train_labels), batch_size=8, shuffle=True, num_workers=2)
    test_loader = DataLoader(CDS(test_feats, test_labels), batch_size=8, num_workers=2)

    import huggingface_hub
    ckpt_path = huggingface_hub.hf_hub_download(repo_id='made-with-clay/Clay', filename='v1.5/clay-v1.5.ckpt')

    class ClaySegmentor(pl.LightningModule):
        def __init__(self):
            super().__init__()
            self.model = Segmentor(num_classes=3, ckpt_path=None)
            self.model.encoder.load_from_ckpt(ckpt_path)
            self.loss_fn = smp.losses.FocalLoss(mode='multiclass')
            from torchmetrics.classification import MulticlassJaccardIndex
            self.iou_per_class = MulticlassJaccardIndex(num_classes=3, average='none')
        def forward(self, batch):
            return self.model({'pixels': batch['pixels'], 'time': batch['time'], 'latlon': batch['latlon'],
                                'gsd': CLAY_GSD, 'waves': CLAY_WAVES})
        def configure_optimizers(self):
            optimizer = torch.optim.AdamW([p for p in self.model.parameters() if p.requires_grad],
                                           lr=1e-5, weight_decay=0.05, betas=(0.9, 0.95))
            scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=100, T_mult=1, eta_min=1e-5 * 100)
            return {'optimizer': optimizer, 'lr_scheduler': {'scheduler': scheduler, 'interval': 'step'}}
        def _shared_step(self, batch, stage):
            labels = batch['label'].long()
            outputs = self(batch)
            if outputs.shape[-2:] != labels.shape[-2:]:
                outputs = F.interpolate(outputs, size=labels.shape[-2:], mode='bilinear', align_corners=False)
            loss = self.loss_fn(outputs, labels)
            self.log(f'{stage}/loss', loss, on_epoch=True, prog_bar=(stage != 'train'))
            return outputs, labels, loss
        def training_step(self, batch, _):
            _, _, loss = self._shared_step(batch, 'train')
            return loss
        def validation_step(self, batch, _):
            self._shared_step(batch, 'val')
        def test_step(self, batch, _):
            outputs, labels, _ = self._shared_step(batch, 'test')
            per_class = self.iou_per_class(outputs, labels)
            for name, val in zip(CLASS_NAMES, per_class):
                self.log(f'test/iou_{name}', val, on_epoch=True)

    model = ClaySegmentor()
    trainer = pl.Trainer(accelerator='gpu', devices=1, precision='bf16-mixed', max_epochs=100,
                          log_every_n_steps=5, callbacks=[pl.callbacks.EarlyStopping(monitor='val/loss', patience=15)],
                          default_root_dir=f'output/p4a_clay_seed{seed}/')
    trainer.fit(model, train_dataloaders=train_loader, val_dataloaders=test_loader)
    metrics = trainer.test(model, dataloaders=test_loader)
    return tidy_terratorch(metrics, 'clay_v1.5', seed)


for seed in [0, 1, 2]:
    for name, fn in [('prithvi', run_prithvi), ('unet', run_unet), ('clay', run_clay)]:
        try:
            print(f'--- running {name} @ seed {seed} ---')
            row = fn(seed)
            print(row)
            results.append(row)
        except Exception as e:
            print(f'FAILED: {name} @ seed {seed}: {e}')
            traceback.print_exc()
            results.append({'model': name, 'seed': seed, 'error': str(e)})
        save_results()

print('=== FINAL ===')
df = pd.DataFrame(results)
print(df.to_string(index=False))

print('=== mean +/- std across seeds ===')
ok = df[~df.get('other', pd.Series(dtype=float)).isna()] if 'other' in df.columns else df
for model_name, grp in ok.groupby('model'):
    for cls in ['other', 'water', 'built_up']:
        if cls in grp.columns:
            vals = grp[cls].dropna()
            if len(vals):
                print(f'{model_name} {cls}: mean={vals.mean():.3f} std={vals.std():.3f} (n={len(vals)})')


## 3. Run it as a subprocess, stream the result back

In [ ]:
import subprocess

proc = subprocess.run(['python', 'run_p4a.py'], capture_output=True, text=True)
print(proc.stdout[-25000:])
print('---stderr (tail)---')
print(proc.stderr[-10000:])
print('returncode:', proc.returncode)
# NOTE: same as P3a/P3b -- no assert on returncode==0. Each (model, seed) failure is
# caught and logged inside run_p4a.py itself.


## 4. Results

In [ ]:
import pandas as pd
comparison = pd.read_csv('comparison_p4a.csv')
print(comparison.to_string(index=False))
print()
for model_name, grp in comparison.groupby('model'):
    for cls in ['other', 'water', 'built_up']:
        if cls in grp.columns:
            vals = grp[cls].dropna()
            if len(vals):
                print(f'{model_name} {cls}: mean={vals.mean():.3f} std={vals.std():.3f} (n={len(vals)})')


## 5. Honest notes for whoever runs this next

- **This cell sequence has not been executed anywhere yet.** It chains 9 full training
  runs (3 models x 3 seeds) in one Kaggle session, reusing P0/P0b's already-verified
  recipes with `pl.seed_everything(seed)` newly added before each run. Same GPU-memory-
  accumulation risk flagged in P3a applies here, now at 9 runs instead of 6.
- **n=3 seeds gives a real mean and std, but not strong statistical power for a formal
  significance test** -- this notebook reports mean+/-std and leaves it there, rather
  than computing a p-value that would overstate the confidence 3 data points can
  support. A proper paired test (e.g. a block bootstrap over the *same held-out tiles*,
  matching H10's own `paired_block_bootstrap_rmse_diff` precedent) would need test_step
  to return per-tile IoU values, not just the batch aggregate -- a real, larger
  engineering change not attempted here.
- **This covers only land cover @ 100% budget.** The 12 remaining configurations (both
  tasks' 25%/50% budgets, flood/SAR's 100% budget) are explicitly out of scope for this
  notebook, not forgotten -- see the intro cell's scope note.
- Once this runs clean (or partially clean): bring back the actual mean+/-std numbers
  and check whether P0's "Prithvi beats U-Net" headline survives real seed-to-seed
  variance, or whether the gap is within noise -- that is the actual question this
  phase exists to answer, not just producing more numbers.


**Update 2026-10-03, real result (Kaggle kernel version 1, COMPLETE, ~5-6 hours, T4
GPU, all 9 runs succeeded with zero failures; all stopped early via EarlyStopping,
range 31-75 epochs -- no crashes, no runaway training):**

| model | class | seed 0 | seed 1 | seed 2 | mean | std |
|---|---|---|---|---|---|---|
| prithvi_eo_v2_300_tl | water | 0.464 | 0.469 | 0.451 | 0.461 | 0.009 |
| prithvi_eo_v2_300_tl | built_up | 0.245 | 0.234 | 0.264 | 0.248 | 0.015 |
| unet_resnet34_scratch | water | 0.316 | 0.337 | 0.330 | 0.328 | 0.011 |
| unet_resnet34_scratch | built_up | 0.200 | 0.205 | 0.211 | 0.205 | 0.006 |
| clay_v1.5 | water | 0.357 | 0.322 | 0.345 | 0.341 | 0.018 |
| clay_v1.5 | built_up | 0.253 | 0.194 | 0.146 | 0.198 | 0.054 |

**Paired per-seed differences (same held-out tiles each seed):**

- Prithvi minus U-Net, water: +0.148, +0.132, +0.121 (mean +0.134, std 0.014) --
  **all three seeds agree on direction**, no overlap near zero. P0's headline claim
  ("Prithvi beats U-Net on water") survives real seed-to-seed variance.
- Prithvi minus U-Net, built_up: +0.045, +0.029, +0.053 (mean +0.042, std 0.012) --
  same pattern, smaller effect but still robust across all three seeds.
- **Clay minus U-Net, water: +0.041, -0.015, +0.015 (mean +0.014, std 0.028)** --
  sign flips across seeds, mean is smaller than its own std. Not robust.
- **Clay minus U-Net, built_up: +0.053, -0.011, -0.065 (mean -0.008, std 0.059)** --
  sign flips across seeds, mean is essentially zero and *negative*.

**This is the real, humbling finding P4 exists to catch.** P0b and P3a's claim that
"Clay beats the from-scratch U-Net on both harder classes" was based on a single
unseeded run (Clay water 0.345 vs U-Net 0.325, built_up 0.224 vs 0.201 -- both
favoring Clay). With real seed variance, that advantage does not hold up: 2 of 3 seeds
actually favor U-Net on both water and built_up, and Clay's own built_up IoU swings
from 0.146 to 0.253 across seeds -- a wider range than the Clay-vs-U-Net difference
itself. **The P0b/P3a "Clay beats U-Net" claim should be read as not robustly
supported, not confirmed.** Meanwhile Prithvi's advantage over U-Net, the P0 headline,
genuinely does survive this check -- all three independent seeds agree on direction
for both classes, with differences well outside the models' own seed-to-seed noise.

With n=3, this is a real mean/std estimate, not a formal significance test (as the
notes above already said it wouldn't attempt) -- but unanimous agreement in sign
across three independent seeds is itself informative, standard practice for flagging
a robust vs. fragile effect even before a larger n. The 12 remaining P4 configurations
(both tasks' 25%/50% budgets, flood/SAR's 100% budget) remain explicitly out of scope
for this notebook.
